## PAKET B

Persiapan Data (Jalankan Terlebih Dahulu)
Buka notebook baru, lalu jalankan kode berikut PERSIS seperti tertulis (jangan diubah) untuk menghasilkan dua
sumber data yang akan anda gunakan sepanjang ujian ini.

In [3]:
import random
import pandas as pd
import json

random.seed(708)
n = 400
lokasi_list = ["Cabang Magelang", "Cabang Semarang", "Cabang Yogyakarta"]
kategori_dirty = ["cuci kering","Cuci Kering","CUCI KERING","setrika",
                "Setrika","cuci setrika","Cuci Setrika"]

pegawai_ids = [f"PG{i:03d}" for i in range(1, 22)]
lokasi_per_pegawai = {pid: random.choice(lokasi_list) for pid in pegawai_ids}

rows = []
for i in range(n):
    pegawai_id = random.choice(pegawai_ids)
    rows.append({
        "cucian_id": f"CCN-{1000+i}",
        "pegawai_id": pegawai_id,
        "cabang": lokasi_per_pegawai[pegawai_id],
        "jenis_layanan": random.choice(kategori_dirty),
        "waktu_proses_jam": random.randint(2, 36),
        "biaya_cucian": random.choice(
            [15000,20000,25000,35000,45000]),
        "status_cucian": random.choice(
        ["Selesai","Selesai","Selesai","Dibatalkan"]),
    })
    
rows += [rows[6], rows[26], rows[92]]
for idx in [8, 44, 150, 192]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_proses_jam"] = None
for idx in [13, 102]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_cucian"] = None
for idx in [232, 282]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_proses_jam"] = -random.randint(1,10)
    
pd.DataFrame(rows).to_csv("cucian.csv", index=False)

pegawai_master = [
    {"pegawai_id": pid, "nama_pegawai": f"Pegawai-{pid}",
    "cabang": lokasi_per_pegawai[pid],
    "rating_pegawai": round(random.uniform(3.0,5.0),1)}
    for pid in pegawai_ids
]
with open("pegawai.json", "w") as f:
    for p in pegawai_master:
        f.write(json.dumps(p) + "\n")
print("Data berhasil dibuat: cucian.csv dan pegawai.json")

Data berhasil dibuat: cucian.csv dan pegawai.json


## SOAL 1. Ekstraksi Data melalui HDFS 

In [11]:
##A. Unggah berkas cucian.csv ke direktori /user/username/uts/raw
!hdfs dfs -mkdir -p /user/haris/uts/raw

!hdfs dfs -ls 

Found 13 items
drwxr-xr-x   - haris supergroup          0 2026-09-12 19:36 ecommerce
drwxr-xr-x   - haris supergroup          0 2026-09-24 16:59 etl_output
drwxr-xr-x   - haris supergroup          0 2026-10-01 12:20 latihan2
drwxr-xr-x   - haris supergroup          0 2026-09-30 23:48 latihan3
drwxr-xr-x   - haris supergroup          0 2026-10-01 12:22 latihanresponsi
drwxr-xr-x   - haris supergroup          0 2026-09-16 17:21 pertemuan4
drwxr-xr-x   - haris supergroup          0 2026-10-01 07:54 pertemuan7
drwxr-xr-x   - haris supergroup          0 2026-09-12 19:27 praktikum
drwxr-xr-x   - haris supergroup          0 2026-09-16 18:18 tugas4
drwxr-xr-x   - haris supergroup          0 2026-09-23 22:47 tugas5
drwxr-xr-x   - haris supergroup          0 2026-09-29 14:31 tugas6
drwxr-xr-x   - haris supergroup          0 2026-10-01 15:51 user
drwxr-xr-x   - haris supergroup          0 2026-10-01 15:54 uts


In [12]:
!hdfs dfs -put cucian.csv /user/haris/uts/raw

In [13]:
!hdfs dfs -ls /user/haris/uts/raw

Found 1 items
-rw-r--r--   1 haris supergroup      25904 2026-10-01 15:54 /user/haris/uts/raw/cucian.csv


In [22]:
import pyspark.sql
##Menyiapkan Spark
    Spark.Session

#B. Baca cucian csv dari hdfs menjadi spark dataframe 
df_cucian = spark.read.csv("https://localhost:9000/user/haris/uts/raw/cucian.csv", header=True, InferSchema=True)

##C. Baca pegawai.json menjadi spark DataFrame
df_pegawai = spark.read.json("pegawai.json", header=True, InferSchema=True)

df_cucian.count()
df_pegawai.count()

df_cucian.PrintSchema()
df_pegawai.PrintSchema()

NameError: name 'spark' is not defined

## SOAL 2. Eksplorasi dan Pembersihan Data 
Data cucian.csv mengandung BEBERAPA masalah kualitas data yang lazim ditemukan pada data dunia nyata.
Lakukan eksplorasi untuk menemukan seluruh masalah tersebut, lalu tangani masing-masing.

In [ ]:
##a. Identifikasi dan tampilkan bukti dari MINIMAL TIGA jenis masalah kualitas data yang berbeda pada cucian.csv.

##b. Tangani seluruh masalah yang ditemukan pada poin (a). Untuk SETIAP penanganan, tuliskan satu kalimat singkat (markdown cell atau komentar kode) yang menjelaskan alasan pendekatan yang dipilih.

##c. Tampilkan jumlah baris data SEBELUM dan SESUDAH proses pembersihan sebagai bukti.

## SOAL 3. Transformasi Data dan Penggabungan

In [29]:
## a. Gabungkan data cucian yang telah bersih dengan data master pegawai, menggunakan kunci penggabungan yang tepat.

## b. Tambahkan kolom baru kategori_sla bernilai ‘Tepat Waktu’ jika waktu_proses_jam ≤ 24, atau ‘Terlambat’ jika sebaliknya.
df_kategori_sla = df.withColumn("kategori_sla") 
    

##c. Tambahkan kolom baru biaya_per_jam_proses, dihitung dari biaya_cucian dibagi waktu_proses_jam.
df_biaya_per_jam = df.withColumn("biaya_per_jam_proses") == col("biaya_cucian") / col("waktu_proses_jam")

##d. Tampilkan 10 baris hasil akhir yang menunjukkan kolom-kolom baru tersebut.
df_biaya_per_jam.show(10)

NameError: name 'df' is not defined

## SOAL 4. Analisis dengan Window Function 

In [ ]:
## a. Untuk SETIAP cabang, tentukan TOP-3 pegawai dengan jumlah cucian berkategori ‘Tepat Waktu’ TERBANYAK. Tampilkan cabang, nama_pegawai, dan jumlahnya.
    df_cucian = df.window
## b. Untuk SETIAP cabang, tentukan SATU pegawai dengan rating TERTINGGI (catatan: gunakan metode pengurutan berbeda dari poin a).

## SOAL 5. Analisis dengan Spark SQL 
Menggunakan Spark SQL murni (spark.sql(), BUKAN DataFrame API), tulis SATU kueri yang menampilkan
cabang, total_cucian, jumlah_terlambat, dan persentase_terlambat (dibulatkan 2 desimal) untuk SETIAP
cabang, diurutkan dari persentase TERTINGGI. Setelah itu, tulis 1-2 kalimat interpretasi: cabang mana yang
paling perlu mendapat perhatian manajemen, dan mengapa?

In [31]:
SELECT cabang, total_cucian, jumlah_terlambat, persentase_terlambat FROM df_gabungan

SyntaxError: invalid syntax (3106508908.py, line 1)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## SOAL 6. Menyimpan Hasil Akhir (ETL Load) 

In [27]:
##a. Simpan hasil akhir Soal 3 (data gabungan yang telah ditransformasi) ke HDFS dalam format Parquet, dipartisi berdasarkan cabang, pada path /user/[username]/uts/processed/laporan_cucian.
!hdfs dfs -ls /user/haris/uts/

df_gabungan = 

!hdfs dfs -put data_gabungan.parquet /user/haris/uts/processed


##b. Verifikasi dengan membaca kembali data tersebut dari HDFS dan menampilkan jumlah barisnya — harus sama dengan jumlah baris hasil Soal 3.
!hdfs dfs -cat data_gabungan.parquet 


SyntaxError: invalid syntax (3360579512.py, line 4)